In [ ]:
import pandas as pd
from pathlib import Path
import re
import glob

In [ ]:
path = Path("data/case_with_all_sources_with_companion_cases_tag.jsonl")
print(path.exists())
print(path.resolve())
df = pd.read_json(path, lines=True)

In [ ]:
df.head(5)

In [ ]:
df = df[df['year'] >= 1975]
df.head()

In [ ]:
df = df[['id', 'year', 'docket_no', 'win_side', 'convos']]

In [ ]:
def get_preceding_advocate(utterances, current_pos, speakers):
    """Walk back to find the last advocate turn, skipping justices."""
    for i in range(current_pos - 1, -1, -1):
        spk = utterances[i].get('speaker_id', '')
        if spk in speakers and speakers[spk]['type'] == 'A':
            return utterances[i]
    return None


rows = []
for _, row in df.iterrows():
    d = row['convos']
    case_id = d.get('case_id', '')
    speakers = d.get('speaker', {})
    win_side = row['win_side']
    docket_no = row['docket_no']
    year = row['year']


    for session in (d.get('utterances') or []):
        if not isinstance(session, list):
            session = [session]

        current_adv_side = None 

        for i, u in enumerate(session):
            spk = u.get('speaker_id', '')

            # Update rolling side tracker for advocates
            if not spk.startswith('j__'):
                if spk in speakers and speakers[spk]['type'] == 'A':
                    current_adv_side = speakers[spk].get('side')
                continue 

            # Walk back to find actual preceding advocate
            prev = get_preceding_advocate(session, i, speakers)
            preceding_speaker = prev['speaker_id'] if prev else None
            preceding_context = prev['text'] if prev else None

            rows.append({
                'case_id': docket_no,
                'year': year,
                'turn_position': i,
                'justice_utterance': u.get('text', ''),
                'preceding_speaker': preceding_speaker,
                'preceding_context': preceding_context,
                'side_addressed': current_adv_side,
                'label': win_side,
            })

df = pd.DataFrame(rows)
df


In [ ]:
df = df[(df['side_addressed'].notna()) & (df['side_addressed'] <= 1) & (df['label'] <= 1)]

In [ ]:
df['label'].value_counts()

In [ ]:
df['side_addressed'].value_counts()

In [ ]:
df.groupby("case_id")["label"].nunique().value_counts()

In [ ]:
df

In [ ]:
df.groupby("case_id").size().value_counts()

In [ ]:
df.head(5)

In [ ]:
print(df.index.is_unique)
print(df['year'].dtype)
print(type(df['year'].iloc[0]))

In [ ]:
print(df['year'].dtype)
print(df['year'].iloc[0])
print(type(df['year'].iloc[0]))

In [ ]:
def balance_split(subset):
    cases = subset.drop_duplicates("case_id")[["case_id", "label"]]
    
    minority_size = cases["label"].value_counts().min()
    
    balanced_cases = (
        cases.groupby("label", group_keys=False)
        .apply(lambda x: x.sample(minority_size, random_state=42))
    )
    
    # Bring back all rows for the selected case_ids
    return subset[subset["case_id"].isin(balanced_cases["case_id"])]


train_df = balance_split(df[df["year"].between(1975, 2004)])
val_df = balance_split(df[df["year"].between(2005, 2009)])
test_df = balance_split(df[df["year"].between(2010, 2019)])

In [ ]:
print(len(train_df['case_id'].unique()))
print(len(test_df['case_id'].unique()))


In [ ]:
train_df.to_csv("data/train_df.csv", index=False)
val_df.to_csv("data/val_df.csv", index=False)
test_df.to_csv("data/test_df.csv", index=False)

In [ ]:
len(df['case_id'].unique())

In [ ]:
df = pd.read_csv('data/train_df.csv')

In [ ]:
df.groupby('case_id')['label'].value_counts()

In [ ]:
for name, split in [("train", train_df), ("val", val_df), ("test", test_df)]:
    counts = split.drop_duplicates("case_id")["label"].value_counts().sort_index()
    is_balanced = counts.nunique() == 1
    print(f"{name}: {counts.to_dict()}")